# English-Spanish NLP Translator

This notebook uses the **Python 3.14 (NLP Translator)** kernel and keeps Hugging Face datasets on disk.

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE_DIR = PROJECT_ROOT / ".hf_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Python:", sys.version.split()[0])
print("Kernel executable:", sys.executable)
print("Dataset cache:", CACHE_DIR)

Python: 3.14.4
Kernel executable: c:\Users\mdsha\AppData\Local\Python\pythoncore-3.14-64\python.exe
Dataset cache: c:\Users\mdsha\Downloads\Natural Language Processing\.hf_cache


In [3]:
from packaging.version import Version
import datasets
from datasets import load_dataset

if Version(datasets.__version__) < Version("5.0.0"):
    raise RuntimeError(
        f"datasets {datasets.__version__} is too old for Python 3.14. "
        "Upgrade datasets and restart the kernel."
    )

nmt_original_valid_set, nmt_test_set = load_dataset(
    "ageron/tatoeba_mt_train",
    "eng-spa",
    split=["validation", "test"],
    cache_dir=str(CACHE_DIR / "datasets"),
    keep_in_memory=False,
)

split = nmt_original_valid_set.train_test_split(
    train_size=0.8,
    seed=42,
    keep_in_memory=False,
)
nmt_train_set = split["train"]
nmt_valid_set = split["test"]

print("datasets version:", datasets.__version__)
print("train rows:", nmt_train_set.num_rows)
print("validation rows:", nmt_valid_set.num_rows)
print("test rows:", nmt_test_set.num_rows)
print("\nFirst nmt_train_set item:")
print(nmt_train_set[0])

datasets version: 5.0.0
train rows: 157839
validation rows: 39460
test rows: 24514

First nmt_train_set item:
{'source_text': 'Tom tried to break up the fight.', 'target_text': 'Tom trató de disolver la pelea.', 'source_lang': 'eng', 'target_lang': 'spa'}


In [4]:
# Load the larger training libraries only after the datasets are ready.
import torch
import torch.nn as nn
import torch.nn.functional as F
import tokenizers
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Translator setup is ready.")

PyTorch: 2.12.0+cu126
CUDA available: True
Translator setup is ready.


In [5]:
nmt_train_set[0]

{'source_text': 'Tom tried to break up the fight.',
 'target_text': 'Tom trató de disolver la pelea.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [7]:
def train_eng_spa():
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

In [9]:
max_length = 200
vocab_size = 10000
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(vocab_size = vocab_size, special_tokens=["<unk>", "<pad>", "<s>", "</s>"])
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)

In [10]:
nmt_tokenizer.encode("I like soccer").ids

[43, 401, 4381]

In [11]:
nmt_tokenizer.encode("<s> Me gusta el futbol").ids

[2, 396, 582, 219, 376, 3075]

In [18]:
from collections import namedtuple

device = "cuda" if torch.cuda.is_available() else "cpu"

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))

In [25]:
def nmt_collate_fn(batch):
    src_texts = [pair["source_text"] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([encodings.ids for encodings in src_encodings], dtype=torch.long)
    tgt_token_ids = torch.tensor([encodings.ids for encodings in tgt_encodings], dtype=torch.long)
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings], dtype=torch.long)
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings], dtype=torch.long)
    inputs = NmtPair(src_token_ids, src_mask, tgt_token_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

In [26]:
batch_size = 32
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn, shuffle=True,
                              pin_memory=(device == "cuda"))
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn,
                              pin_memory=(device == "cuda"))
nmt_test_loader = DataLoader(nmt_test_set, batch_size=batch_size,
                             collate_fn=nmt_collate_fn,
                             pin_memory=(device == "cuda"))


In [29]:
class NmtModel(nn.Module):
    def __init__(self, vocab_size, hidden_size = 512, num_layers = 2, embed_dim = 512):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.encoder = nn.GRU(embed_dim, hidden_size=hidden_size, batch_first=True, num_layers = num_layers)
        self.decoder = nn.GRU(embed_dim, hidden_size = hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)
    def forward(self, pair):
        src_embeddings = self.embeddings(pair.src_token_ids)
        tgt_embeddings = self.embeddings(pair.tgt_token_ids)
        lengths = pair.src_mask.sum(dim = 1)
        padded_sequence = pack_padded_sequence(src_embeddings, lengths=lengths.cpu(), enforce_sorted=False, batch_first=True)
        _, hidden_states = self.encoder(padded_sequence)
        outputs, hidden_states = self.decoder(tgt_embeddings, hidden_states)
        return self.output(outputs).permute(0,2,1)

torch.manual_seed(42)
vocab_size = nmt_tokenizer.get_vocab_size()
nmt_model = NmtModel(vocab_size).to(device = "cuda")

In [30]:
import torchmetrics

def evaluate_tm(model, valid_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs, patience = 2, factor = 0.5, epoch_callback = None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = "max", patience=patience, factor = factor)
    history = {"train_losses" : [], "train_metrics" : [], "valid_metrics" : []}
    for epoch in range(n_epochs):
        total_loss = 0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1} / {len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history

n_epochs = 10
xentropy = nn.CrossEntropyLoss(ignore_index=0)  # ignore <pad> tokens
optimizer = torch.optim.NAdam(nmt_model.parameters(), lr=0.001)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size)
accuracy = accuracy.to(device)

history = train(nmt_model, optimizer, xentropy, accuracy,
                nmt_train_loader, nmt_valid_loader, n_epochs)

Epoch 1/10,                      train loss: 3.5383, train metric: 12.94%, valid metric: 15.74%
Epoch 2/10,                      train loss: 2.2663, train metric: 17.48%, valid metric: 16.65%
Epoch 3/10,                      train loss: 1.9010, train metric: 19.00%, valid metric: 16.81%
Epoch 4/10,                      train loss: 1.7177, train metric: 19.85%, valid metric: 16.82%
Epoch 5/10,                      train loss: 1.6171, train metric: 20.23%, valid metric: 16.74%
Epoch 6/10,                      train loss: 1.5621, train metric: 20.43%, valid metric: 16.62%
Epoch 7/10,                      train loss: 1.5340, train metric: 20.47%, valid metric: 16.55%
Epoch 8/10,                      train loss: 1.1882, train metric: 22.64%, valid metric: 17.37%
Epoch 9/10,                      train loss: 0.9315, train metric: 24.30%, valid metric: 17.38%
Epoch 10/10,                      train loss: 0.8096, train metric: 25.17%, valid metric: 17.36%


In [52]:
def translate(model, src_text, max_length=20, start_id=None, eos_id=None):
    """Greedy autoregressive decoding using the existing model weights."""
    model_device = next(model.parameters()).device
    start_id = nmt_tokenizer.token_to_id("<s>") if start_id is None else start_id
    eos_id = nmt_tokenizer.token_to_id("</s>") if eos_id is None else eos_id

    src_encoding = nmt_tokenizer.encode(src_text)
    src_token_ids = torch.tensor([src_encoding.ids], dtype=torch.long, device=model_device)
    src_mask = torch.tensor([src_encoding.attention_mask], dtype=torch.long, device=model_device)
    generated_ids = [start_id]

    model.eval()
    with torch.inference_mode():
        for _ in range(max_length):
            tgt_token_ids = torch.tensor([generated_ids], dtype=torch.long, device=model_device)
            tgt_mask = torch.ones_like(tgt_token_ids)
            pair = NmtPair(src_token_ids, src_mask, tgt_token_ids, tgt_mask)
            logits = model(pair)
            next_token_id = int(logits[0, :, -1].argmax().item())
            if next_token_id == eos_id:
                break
            generated_ids.append(next_token_id)

    return nmt_tokenizer.decode(generated_ids[1:], skip_special_tokens=True)

In [32]:
def attention(query, key, value, key_mask=None):
    scale = query.size(-1) ** -0.5
    weights = (query @ key.transpose(1, 2)) * scale
    if key_mask is not None:
        weights = weights.masked_fill(~key_mask[:, None, :].bool(), float("-inf"))
    softmax_weights = torch.softmax(weights, dim=-1)
    return softmax_weights @ value

In [41]:
class NmtModelWithAttention(nn.Module):
    def __init__(self, vocab_size, hidden_size = 512, embed_dim = 512, num_layers = 2, pad_id = 0):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.encoder = nn.GRU(embed_dim, hidden_size, num_layers=num_layers, batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden_size, num_layers=num_layers, batch_first=True)
        self.outputs = nn.Linear(2 * hidden_size, vocab_size)
    def forward(self, pair):
        encoder_embeddings = self.embeddings(pair.src_token_ids)
        decoder_embeddings = self.embeddings(pair.tgt_token_ids)
        lengths = pair.src_mask.sum(dim = 1)
        packed_sequence = pack_padded_sequence(encoder_embeddings, lengths=lengths.cpu(), enforce_sorted=False, batch_first=True)
        encoder_outputs_packed, hidden_states_encoder = self.encoder(packed_sequence)
        decoder_outputs, hidden_states_decoder = self.decoder(decoder_embeddings, hidden_states_encoder)
        padded_sequence, _ = pad_packed_sequence(encoder_outputs_packed, batch_first=True)
        encoder_mask = pair.src_mask[:, :padded_sequence.size(1)]
        attention_vector = attention(query=decoder_outputs, key=padded_sequence,
                                     value=padded_sequence, key_mask=encoder_mask)
        combined_outputs = torch.cat((attention_vector, decoder_outputs), dim = -1)
        return self.outputs(combined_outputs).permute(0,2,1)

In [53]:
torch.manual_seed(42)
best_checkpoint = PROJECT_ROOT / "checkpoints" / "nmt_attention_best.pt"

if best_checkpoint.exists():
    checkpoint = torch.load(best_checkpoint, map_location=device, weights_only=False)
    nmt_tokenizer = tokenizers.Tokenizer.from_str(checkpoint["tokenizer_json"])
    vocab_size = checkpoint["vocab_size"]
    nmt_attn_model = NmtModelWithAttention(vocab_size).to(device)
    nmt_attn_model.load_state_dict(checkpoint["model_state_dict"])
    nmt_attn_model.eval()
    history = checkpoint.get("history", {})
    print(f"Loaded trained attention model from {best_checkpoint}")
    print(f"Best validation token accuracy: {checkpoint['best_accuracy']:.2%} "
          f"(epoch {checkpoint['epoch'] + 1})")
else:
    nmt_attn_model = NmtModelWithAttention(vocab_size).to(device)
    n_epochs = 10
    xentropy = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = torch.optim.NAdam(nmt_attn_model.parameters(), lr=0.001)
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size).to(device)
    history = train(nmt_attn_model, optimizer, xentropy, accuracy,
                    nmt_train_loader, nmt_valid_loader, n_epochs)

Loaded trained attention model from checkpoints\nmt_attention_best.pt
Best validation token accuracy: 66.72% (epoch 5)


In [54]:
translation_tests = [
    "Hello, I like playing football.",
    "Where is the train station?",
    "This is a beautiful day.",
    "I love you.",
    "Can you help me?",
    "What time is it?",
]
for text in translation_tests:
    print(f"{text} -> {translate(nmt_attn_model, text, max_length=40)}")

Hello, I like playing football. -> Buenos días me gusta jugar al fútbol .
Where is the train station? -> ¿ Dónde está la estación de tren ?
This is a beautiful day. -> Este es un día hermoso .
I love you. -> Te quiero .
Can you help me? -> ¿ Puedes ayudarme ?
What time is it? -> ¿ A qué hora es ?


In [ ]:
def load_attention_model(checkpoint_path=PROJECT_ROOT / "checkpoints" / "nmt_attention_best.pt"):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    tokenizer = tokenizers.Tokenizer.from_str(checkpoint["tokenizer_json"])
    model = NmtModelWithAttention(checkpoint["vocab_size"]).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, tokenizer

# In a fresh kernel, run the setup/model-definition cells, then uncomment:
# nmt_attn_model, nmt_tokenizer = load_attention_model()